## Project #2. Scheduling the NBA ##

The National Basketball Association (NBA) is planning their schedule for the 2025/2026 season. In the attached games.csv you can find a (ficticious) preliminary schedule Among other data, you find, for each match, the home team, the away team, and the date and location of the match. The goal of the project is to possibly improve the current draft of the schedule, under some of the constraints imposed by the current schedule.

The project is composed of 4 questions, all to be solved using Integer Programming, to be completed in this order:

1. Write codes that computes and prints, for each team i, the following information:

(a) all the dates when team i played home; (b) for each team j, the number of times team i played against team j at home; (c) for each team j, the number of times team i played against team j away (i.e., at j's home); (d) all the dates when team j played away.

2. Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed in (a) above; (f) plays away (possibly, against a different team) exactly on the dates computed in (d) above; (g) for each team j, i plays home against team j exactly the number of times computed in (b) above; (h) for each team j, i plays away against team j (i.e., at j's home) exactly the number of times computed in (c) above.

3. Compute a feasible schedule that satisfy the following additional constraint: no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more; or conclude that no such schedule exists. For instance, if a team plays game 1,2,3 and:
- the time zone difference between the arena where game 1 is played and the arena where game 2 is played is 2;
- the time zone difference between the arena where game 2 is played and the arena where game 3 is played is 3;
Then the schedue is infeasible, since 3+2 = 5 $\geq$ 4.

4. **This part will not be graded, but we may ask you about it in the one-on-one discussion** Compute any improvement to the current schedule. For instance, using Google Maps, you could compute and store the locations of all arenas, compute a feasible schedule that minimizes the maximum distance traveled by a team, and compare this value with the one of the current schedule to show the improvement. You can also access the full 2023/2024 season schedule at https://www.basketball-reference.com/leagues/NBA_2024_games-october.html.

**Deliverables**
- ipynb file containing all the code
- pdf file explaining the models, the variable, and the constraints
- Schedules saved in a .csv or analagous file, presenting the list of matches organized as follows: (a) date (b) team playing home (c) team playing away (d)


In [28]:
import pandas as pd
from collections import defaultdict
from pulp import LpProblem, LpVariable, LpBinary, lpSum, LpStatus

## Problem 1: Extracting Structural Scheduling Constraints ##

Write codes that computes and prints, for each team i, the following information:

(a) all the dates when team i played home; 
(b) for each team j, the number of times team i played against team j at home; 
(c) for each team j, the number of times team i played against team j away (i.e., at j's home); 
(d) all the dates when team j played away.

In [ ]:
print("\n------------------------------------------------------------")
print("Problem 1: Extracting Structural Scheduling Constraints")
print("------------------------------------------------------------\n")

# -----------------------------
# Load the schedule dataset
# -----------------------------
path = "/Users/athena/Desktop/IEOR4004E OPTIMIZATION MODELS AND METHODS/Project 2/games.xlsx"   # Change this path if needed
games = pd.read_excel(path)

# Rename columns for convenience
games = games.rename(columns={
    "Visitor": "AwayTeam",
    "Home": "HomeTeam",
    "Date": "Date"
})

games["Date"] = pd.to_datetime(games["Date"])

# Convert date to datetime format
games["Date"] = pd.to_datetime(games["Date"])

# List of all teams
teams = sorted(set(games["HomeTeam"]).union(set(games["AwayTeam"])))
print("Total number of teams:", len(teams))


# -------------------------------------------------------
# For each team i, print:
# (a) all home dates
# (b) number of times i plays HOME vs each team j
# (c) number of times i plays AWAY vs each team j
# (d) all away dates
# -------------------------------------------------------

for team in teams:
    print("="*70)
    print("Team:", team)

    # ------------------------------------------------------------
    # (a) All dates when team i played at HOME
    # ------------------------------------------------------------
    home_games = games[games["HomeTeam"] == team].sort_values("Date")
    home_dates = list(home_games["Date"].dt.date.unique())

    print("(a) Home dates:")
    print("   ", home_dates)

    # ------------------------------------------------------------
    # (d) All dates when team i played AWAY
    # ------------------------------------------------------------
    away_games = games[games["AwayTeam"] == team].sort_values("Date")
    away_dates = list(away_games["Date"].dt.date.unique())

    print("(d) Away dates:")
    print("   ", away_dates)

    # ------------------------------------------------------------
    # (b) For each opponent j: number of HOME games vs j
    # ------------------------------------------------------------
    home_vs_count = defaultdict(int)
    for _, row in home_games.iterrows():
        opp = row["AwayTeam"]
        home_vs_count[opp] += 1

    print("(b) Number of HOME games vs each opponent:")
    for opp in sorted(home_vs_count.keys()):
        print(f"     vs {opp}: {home_vs_count[opp]}")

    # ------------------------------------------------------------
    # (c) For each opponent j: number of AWAY games vs j
    # ------------------------------------------------------------
    away_vs_count = defaultdict(int)
    for _, row in away_games.iterrows():
        opp = row["HomeTeam"]
        away_vs_count[opp] += 1

    print("(c) Number of AWAY games @ each opponent:")
    for opp in sorted(away_vs_count.keys()):
        print(f"     @ {opp}: {away_vs_count[opp]}")

print("\nQuestion 1 completed.")


------------------------------------------------------------
Problem 1: Extracting Structural Scheduling Constraints
------------------------------------------------------------

Total number of teams: 16
Team: Atlanta Hawks
(a) Home dates:
    [datetime.date(2025, 11, 3), datetime.date(2025, 11, 7), datetime.date(2025, 11, 15), datetime.date(2025, 11, 17), datetime.date(2025, 11, 19), datetime.date(2025, 11, 23), datetime.date(2025, 11, 27), datetime.date(2025, 11, 28), datetime.date(2025, 11, 29), datetime.date(2025, 12, 25)]
(d) Away dates:
    [datetime.date(2025, 11, 1), datetime.date(2025, 11, 5), datetime.date(2025, 11, 11), datetime.date(2025, 11, 13), datetime.date(2025, 11, 21), datetime.date(2025, 12, 1)]
(b) Number of HOME games vs each opponent:
     vs Boston Celtics: 1
     vs Chicago Bulls: 1
     vs Dallas Mavericks: 1
     vs Golden State Warriors: 1
     vs Los Angeles Lakers: 1
     vs Miami Heat: 1
     vs Milwaukee Bucks: 1
     vs New York Knicks: 1
     vs Phoe

In [33]:
# ----------------------------------------------------------
# Rebuild home_dates and away_dates using team names as keys
# ----------------------------------------------------------

home_dates = {team: [] for team in teams}
away_dates = {team: [] for team in teams}

for _, row in games.iterrows():
    date = row["Date"].date()   # convert Timestamp → datetime.date

    home = row["HomeTeam"]
    away = row["AwayTeam"]

    home_dates[home].append(date)
    away_dates[away].append(date)

# sort
for t in teams:
    home_dates[t] = sorted(home_dates[t])
    away_dates[t] = sorted(away_dates[t])
    away_dates[t] = sorted(away_dates[t])

## Problem 2: Integer Programming Model for Feasible Schedules ##

Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed in (a) above; 
(f) plays away (possibly, against a different team) exactly on the dates computed in (d) above; 
(g) for each team j, i plays home against team j exactly the number of times computed in (b) above; (h) for each team j, i plays away against team j (i.e., at j's home) exactly the number of times computed in (c) above.

In [35]:

print("\n------------------------------------------------------------")
print("Problem 2: Integer Programming Model for Feasible Schedules")
print("------------------------------------------------------------\n")

# ------------------------------------------------------------
# Build dates list
# ------------------------------------------------------------
dates = sorted(games["Date"].dt.date.unique())

# ------------------------------------------------------------
# Build dictionaries from Q1
# ------------------------------------------------------------
home_dates_dict = {
    team: list(games[games["HomeTeam"] == team]["Date"].dt.date.unique())
    for team in teams
}

away_dates_dict = {
    team: list(games[games["AwayTeam"] == team]["Date"].dt.date.unique())
    for team in teams
}

# Home/away matchup frequencies
home_vs = { (i,j): 0 for i in teams for j in teams if i != j }
away_vs = { (i,j): 0 for i in teams for j in teams if i != j }

for _, row in games.iterrows():
    i = row["HomeTeam"]
    j = row["AwayTeam"]
    home_vs[(i,j)] += 1
    away_vs[(j,i)] += 1     # j plays away at i


# ------------------------------------------------------------
# Decision Variables:
# x[i,j,d] = 1 if on date d, team i plays HOME vs team j
# ------------------------------------------------------------
model = LpProblem("NBA_Feasible_Schedule")   # no objective function

x = {}
for i in teams:
    for j in teams:
        if i != j:
            for d in dates:
                x[(i,j,d)] = LpVariable(f"x_{i}_{j}_{d}", cat=LpBinary)


# ------------------------------------------------------------
# Constraint (e): Home dates must match Q1
# ------------------------------------------------------------
for i in teams:
    for d in dates:
        if d not in home_dates_dict[i]:
            model += lpSum(x[(i,j,d)] for j in teams if i != j) == 0


# ------------------------------------------------------------
# Constraint (f): Away dates must match Q1
# ------------------------------------------------------------
for i in teams:
    for d in dates:
        if d not in away_dates_dict[i]:
            model += lpSum(x[(j,i,d)] for j in teams if i != j) == 0


# ------------------------------------------------------------
# One game per team per date
# ------------------------------------------------------------
for i in teams:
    for d in dates:
        model += (
            lpSum(x[(i,j,d)] for j in teams if i != j) +
            lpSum(x[(j,i,d)] for j in teams if i != j)
        ) <= 1


# ------------------------------------------------------------
# Constraint (g): home vs j count must match Q1
# ------------------------------------------------------------
for i in teams:
    for j in teams:
        if i != j:
            model += lpSum(x[(i,j,d)] for d in dates) == home_vs[(i,j)]


# ------------------------------------------------------------
# Constraint (h): away @ j count must match Q1
# ------------------------------------------------------------
for i in teams:
    for j in teams:
        if i != j:
            model += lpSum(x[(j,i,d)] for d in dates) == away_vs[(i,j)]


# ------------------------------------------------------------
# Solve the feasibility model
# ------------------------------------------------------------
print("Solving feasibility model ...")
model.solve()

print("Model status:", LpStatus[model.status])

if LpStatus[model.status] == "Optimal":
    print("A feasible schedule exists!")
else:
    print("No feasible schedule exists.")


------------------------------------------------------------
Problem 2: Integer Programming Model for Feasible Schedules
------------------------------------------------------------

Solving feasibility model ...
Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Mar  5 2025 

command line - cbc /var/folders/zj/z2hzy0s92f985pcv9c_k1x0m0000gn/T/03018d101c004c638f2da0d79d9755b5-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/zj/z2hzy0s92f985pcv9c_k1x0m0000gn/T/03018d101c004c638f2da0d79d9755b5-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 997 COLUMNS
At line 27879 RHS
At line 28872 BOUNDS
At line 32714 ENDATA
Problem MODEL has 992 rows, 3841 columns and 19200 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.01 seconds
Cgl0002I 3263 variables fixed
Cgl0003I 0 fixed, 0 tightened bounds, 0 strengthened rows, 5 substitutions
Cgl0004I pr

In [36]:
# ------------------------------------------------------------
# Export the feasible schedule found in Question 2 into a CSV file (date, home team, away team)
# ------------------------------------------------------------


print("\n------------------------------------------------------------")
print("Exporting feasible schedule to CSV...")
print("------------------------------------------------------------\n")

# Collect all scheduled games from variable x[(i,j,d)]
schedule_rows = []

for (i, j, d), var in x.items():
    if var.value() == 1:
        schedule_rows.append({
            "Date": d,
            "HomeTeam": i,
            "AwayTeam": j
        })

# Convert to DataFrame
schedule_df = pd.DataFrame(schedule_rows)

# Sort schedule by date (and optionally by home team)
schedule_df = schedule_df.sort_values(by=["Date", "HomeTeam"]).reset_index(drop=True)

# Save to CSV
output_path = "feasible_schedule.csv"
schedule_df.to_csv(output_path, index=False)

print(f"Feasible schedule exported successfully to: {output_path}")
schedule_df.head()


------------------------------------------------------------
Exporting feasible schedule to CSV...
------------------------------------------------------------

Feasible schedule exported successfully to: feasible_schedule.csv


,Date,HomeTeam,AwayTeam
0,2025-11-01,Boston Celtics,Dallas Mavericks
1,2025-11-01,Brooklyn Nets,Atlanta Hawks
2,2025-11-01,Chicago Bulls,Denver Nuggets
3,2025-11-01,Miami Heat,Houston Rockets
4,2025-11-01,Milwaukee Bucks,Cleveland Cavaliers


## Problem 3: Time-Zone-Constrained Schedule Feasibility ##

Compute a feasible schedule that satisfy the following additional constraint: no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more; or conclude that no such schedule exists. For instance, if a team plays game 1,2,3 and:
- the time zone difference between the arena where game 1 is played and the arena where game 2 is played is 2;
- the time zone difference between the arena where game 2 is played and the arena where game 3 is played is 3;
Then the schedue is infeasible, since 3+2 = 5 $\geq$ 4.

In [37]:
import itertools
from pulp import LpProblem, LpVariable, LpMinimize, LpStatus, lpSum, LpBinary

In [38]:
# -------------------------------------------------------
# Time zone mapping for all NBA arenas (Problem 3 required)
# -------------------------------------------------------
team_timezones = {
    # Eastern Time (UTC-5)
    "Boston Celtics": -5,
    "Brooklyn Nets": -5,
    "New York Knicks": -5,
    "Philadelphia 76ers": -5,
    "Toronto Raptors": -5,
    "Cleveland Cavaliers": -5,
    "Detroit Pistons": -5,
    "Indiana Pacers": -5,
    "Miami Heat": -5,
    "Orlando Magic": -5,
    "Washington Wizards": -5,
    "Atlanta Hawks": -5,
    "Charlotte Hornets": -5,
    "Chicago Bulls": -6,   # Chicago is Central
    "Milwaukee Bucks": -6, # Central
    # Western Conference → Central Time first
    "Houston Rockets": -6,
    "Dallas Mavericks": -6,
    "San Antonio Spurs": -6,
    "New Orleans Pelicans": -6,
    "Memphis Grizzlies": -6,
    "Oklahoma City Thunder": -6,
    "Minnesota Timberwolves": -6,

    # Mountain Time (UTC-7)
    "Denver Nuggets": -7,
    "Utah Jazz": -7,
    "Phoenix Suns": -7,

    # Pacific Time (UTC-8)
    "Golden State Warriors": -8,
    "Los Angeles Lakers": -8,
    "Los Angeles Clippers": -8,
    "Portland Trail Blazers": -8,
    "Sacramento Kings": -8
}



In [ ]:

print("\n------------------------------------------------------------")
print("Problem 3: Checking feasibility under time-zone constraints")
print("------------------------------------------------------------\n")


# NOTE:
# - We assume variable x[(i,j,d)] from Problem 2 is already defined
# - We assume feasible schedule from Problem 2 is stored in feasible_schedule.csv
# - We only ADD the time-zone constraint to check feasibility again

# -----------------------------------------
# Load feasible schedule from Problem 2
# -----------------------------------------

schedule_df = pd.read_csv("feasible_schedule.csv")

# Build a dictionary mapping each team to their (date, location timezone)
# Problem 3 requires knowing the time zone of each arena
# Assume arena_timezones[team] already defined in previous part:
# Example: arena_timezones = {"Lakers": -8, "Knicks": -5, ...}

# Convert dates to sortable objects
schedule_df["Date"] = pd.to_datetime(schedule_df["Date"])
schedule_df = schedule_df.sort_values("Date")

# -----------------------------------------
# Build mapping: team -> list of (date, timezone)
# -----------------------------------------

team_games = {team: [] for team in teams}

for _, row in schedule_df.iterrows():
    d = row["Date"]
    
    home = row["HomeTeam"]
    away = row["AwayTeam"]

    home_loc = team_timezones[home]
    away_loc = team_timezones[away]
    
    # Home team location
    team_games[home].append((d, home_loc))
    
    # Away team location = the home team's timezone
    team_games[away].append((d, away_loc))


# Sort each team's games by date
for t in teams:
    team_games[t] = sorted(team_games[t], key=lambda x: x[0])

# -------------------------------------------------------
# Build NEW IP model including time-zone constraints
# -------------------------------------------------------

model3 = LpProblem("NBA_TimeZone_Constrained_Schedule", LpMinimize)

# Reuse decision variables x[(i,j,d)] from Problem 2
# But PuLP requires variables inside the model; we reconstruct x3 here.

x3 = LpVariable.dicts("x",
                      ((i, j, d) for i in teams for j in teams for d in dates if i != j),
                      lowBound=0, upBound=1, cat=LpBinary)

# -------------------------------------------------------
# Constraint group 1–3: Same as Problem 2
# -------------------------------------------------------

# (1) Home-date constraints  —— corrected
for i in teams:
    for d in dates:
        # NOTE: home_dates[i] is a list of dates when i plays home
        if d in home_dates[i]:
            model3 += lpSum(x3[(i, j, d)] for j in teams if j != i) == 1
        else:
            model3 += lpSum(x3[(i, j, d)] for j in teams if j != i) == 0

# (2) Away-date constraints  —— corrected
for i in teams:
    for d in dates:
        # away_dates[i] is a list of dates when i plays AWAY
        if d in away_dates[i]:
            model3 += lpSum(x3[(j, i, d)] for j in teams if j != i) == 1
        else:
            model3 += lpSum(x3[(j, i, d)] for j in teams if j != i) == 0

# Recompute home_counts and away_counts from original games
home_counts = {i: {j: 0 for j in teams} for i in teams}
away_counts = {i: {j: 0 for j in teams} for i in teams}

for _, row in games.iterrows():
    home_team = row["HomeTeam"]
    away_team = row["AwayTeam"]
    home_counts[home_team][away_team] += 1
    away_counts[away_team][home_team] += 1
    
# (3) Opponent frequency (home/away)
for i in teams:
    for j in teams:
        if i != j:
            model3 += lpSum(x3[(i,j,d)] for d in dates) == home_counts[i][j]
            model3 += lpSum(x3[(j,i,d)] for d in dates) == away_counts[i][j]

# -------------------------------------------------------
# Constraint group 4: Time-Zone Consecutive Games Constraint
# -------------------------------------------------------
# For each team i, for each consecutive triple of dates d1 < d2 < d3:
# Let tz(d) be the arena timezone for that game.
# Require:
#   |tz(d2) - tz(d1)| + |tz(d3) - tz(d2)| <= 3
#
# We linearize absolute values using standard Big-M formulations.

M = 1000  # Big-M

for team in teams:
    games = team_games[team]  # list of (date, timezone)

    # Need at least 3 games to check triples
    if len(games) < 3:
        continue

    for k in range(len(games) - 2):
        d1, tz1 = games[k]
        d2, tz2 = games[k+1]
        d3, tz3 = games[k+2]

        # Compute absolute tz jumps
        jump1 = abs(tz2 - tz1)
        jump2 = abs(tz3 - tz2)

        # If jump1 + jump2 >= 4, prohibit this triple from happening
        if jump1 + jump2 >= 4:
            # Add constraint forbidding this triple:
            # A team cannot play all three games with x3 assignment matching (date, opponent)
            # Simplest: forbid the triple from occurring:
            #
            # For home game:
            cond1 = lpSum(x3[(team, j, d1.date())] for j in teams if j != team)
            cond2 = lpSum(x3[(team, j, d2.date())] for j in teams if j != team)
            cond3 = lpSum(x3[(team, j, d3.date())] for j in teams if j != team)

            # For away game:
            cond1b = lpSum(x3[(j, team, d1.date())] for j in teams if j != team)
            cond2b = lpSum(x3[(j, team, d2.date())] for j in teams if j != team)
            cond3b = lpSum(x3[(j, team, d3.date())] for j in teams if j != team)

            # Team must not play all three of these high-jump games
            model3 += cond1 + cond1b + cond2 + cond2b + cond3 + cond3b <= 2

# -------------------------------------------------------
# Solve Problem 3
# -------------------------------------------------------

print("Solving Problem 3 (timezone constraints)...")
status3 = model3.solve()
print("Model 3 Status:", LpStatus[status3])

if LpStatus[status3] != "Optimal":
    print("\nNo feasible schedule satisfies the time-zone constraints.")
else:
    print("\nA feasible schedule satisfying time-zone constraints exists!")

    # Export the new feasible schedule
    schedule3_rows = []
    for (i,j,d), var in x3.items():
        if var.value() == 1:
            schedule3_rows.append({"Date": d, "HomeTeam": i, "AwayTeam": j})

    schedule3_df = pd.DataFrame(schedule3_rows)
    schedule3_df = schedule3_df.sort_values(["Date", "HomeTeam"])
    schedule3_df.to_csv("feasible_schedule_problem3.csv", index=False)

    print("Exported as feasible_schedule_problem3.csv")


------------------------------------------------------------
Problem 3: Checking feasibility under time-zone constraints
------------------------------------------------------------

Solving Problem 3 (timezone constraints)...
Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Mar  5 2025 

command line - cbc /var/folders/zj/z2hzy0s92f985pcv9c_k1x0m0000gn/T/4e64ede11b824d7da1f5942a6ffad29f-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/zj/z2hzy0s92f985pcv9c_k1x0m0000gn/T/4e64ede11b824d7da1f5942a6ffad29f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 997 COLUMNS
At line 24039 RHS
At line 25032 BOUNDS
At line 28874 ENDATA
Problem MODEL has 992 rows, 3841 columns and 15360 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.01 seconds
Cgl0002I 3263 variables fixed
Cgl0004I processed model has 320 rows, 454 columns (454 integer (454 of 

## Problem 4: Schedule Improvement via Travel Optimization ##

**This part will not be graded, but we may ask you about it in the one-on-one discussion** Compute any improvement to the current schedule. For instance, using Google Maps, you could compute and store the locations of all arenas, compute a feasible schedule that minimizes the maximum distance traveled by a team, and compare this value with the one of the current schedule to show the improvement. You can also access the full 2023/2024 season schedule at https://www.basketball-reference.com/leagues/NBA_2024_games-october.html.

In [ ]:

print("\n------------------------------------------------------------")
print("Problem 4: Schedule Improvement via Travel Optimization (proxy)")
print("------------------------------------------------------------\n")

# We treat time-zone differences as a proxy for travel distance.
# We compare the original preliminary schedule (games) with the time-zone-constrained schedule from Problem 3 stored in feasible_schedule_problem3.csv.

# ------------------------------------------------
# Load original schedule from Problem 1 (games.csv)
# ------------------------------------------------
games = pd.read_csv("games.csv")

orig = games.copy()

# Original column names: Date, Home, Visitor
orig["Date"] = pd.to_datetime(orig["Date"])

# Rename to unify naming with feasible schedules from Problem 3
orig = orig.rename(columns={
    "Home": "HomeTeam",
    "Visitor": "AwayTeam"
})

# Keep only needed columns
orig = orig[["Date", "HomeTeam", "AwayTeam"]]

# Add time zone based on home team
orig["TZ"] = orig["HomeTeam"].map(team_timezones)

# ------------------------------------------------
# Load improved schedule from Problem 3
# ------------------------------------------------
imp = pd.read_csv("feasible_schedule_problem3.csv")
imp["Date"] = pd.to_datetime(imp["Date"])
imp = imp.sort_values("Date")

# Add time zone (home team's timezone)
imp["TZ"] = imp["HomeTeam"].map(team_timezones)

# ------------------------------------------------
# Function to compute total travel burden for a team schedule
# ------------------------------------------------
def compute_travel_burden(df, team):
    # Extract games played by the team (home or away)
    g = df[(df["HomeTeam"] == team) | (df["AwayTeam"] == team)]
    g = g.sort_values("Date")

    burden = 0
    forbidden = 0

    for k in range(len(g) - 2):
        tz1 = g.iloc[k]["TZ"]
        tz2 = g.iloc[k+1]["TZ"]
        tz3 = g.iloc[k+2]["TZ"]

        jump = abs(tz2 - tz1) + abs(tz3 - tz2)

        burden += jump
        if jump >= 4:
            forbidden += 1

    return burden, forbidden

# ------------------------------------------------
# Evaluate original vs improved schedule
# ------------------------------------------------
teams_list = list(team_timezones.keys())
orig_total = {}
imp_total = {}
orig_bad = {}
imp_bad = {}

for team in teams_list:
    b1, f1 = compute_travel_burden(orig, team)
    b2, f2 = compute_travel_burden(imp, team)

    orig_total[team] = b1
    imp_total[team] = b2
    orig_bad[team] = f1
    imp_bad[team] = f2


# ------------------------------------------------
# Summary
# ------------------------------------------------
rows = []

for team in teams_list:
    o1 = orig_total[team]
    i1 = imp_total[team]
    o2 = orig_bad[team]
    i2 = imp_bad[team]

    print(f"{team}")
    print(f"    TZ burden:           {o1} → {i1}")
    print(f"    Forbidden triples:    {o2} → {i2}\n")

    rows.append([team, o1, i1, o2, i2])

# ---- summary values ----
avg_orig_tz = sum(orig_total.values()) / len(teams_list)
avg_imp_tz  = sum(imp_total.values()) / len(teams_list)
sum_orig_bad = sum(orig_bad.values())
sum_imp_bad  = sum(imp_bad.values())

print("----------------------------------------------------------")
print("Overall Summary:")
print(f"    Average TZ burden:       {avg_orig_tz:.2f} → {avg_imp_tz:.2f}")
print(f"    Total forbidden triples: {sum_orig_bad} → {sum_imp_bad}")
print("==========================================================\n")

# ---- Write to CSV ----
rows.append([
    "AVERAGE",
    round(avg_orig_tz, 2),
    round(avg_imp_tz, 2),
    sum_orig_bad,
    sum_imp_bad
])

df_result = pd.DataFrame(rows, columns=[
    "Team",
    "TZ_Original",
    "TZ_Improved",
    "Forbidden_Original",
    "Forbidden_Improved"
])

df_result.to_csv("schedule_improvement_summary_problem4.csv", index=False)
print("Summary exported to CSV: schedule_improvement_summary_problem4.csv")



------------------------------------------------------------
Problem 4: Schedule Improvement via Travel Optimization (proxy)
------------------------------------------------------------

Boston Celtics
    TZ burden:           20 → 31
    Forbidden triples:    1 → 3

Brooklyn Nets
    TZ burden:           25 → 28
    Forbidden triples:    1 → 2

New York Knicks
    TZ burden:           25 → 34
    Forbidden triples:    2 → 3

Philadelphia 76ers
    TZ burden:           21 → 25
    Forbidden triples:    1 → 1

Toronto Raptors
    TZ burden:           21 → 7
    Forbidden triples:    2 → 0

Cleveland Cavaliers
    TZ burden:           25 → 28
    Forbidden triples:    2 → 3

Detroit Pistons
    TZ burden:           0 → 0
    Forbidden triples:    0 → 0

Indiana Pacers
    TZ burden:           0 → 0
    Forbidden triples:    0 → 0

Miami Heat
    TZ burden:           13 → 15
    Forbidden triples:    1 → 1

Orlando Magic
    TZ burden:           0 → 0
    Forbidden triples:    0 → 0

Was